# Experiment 37 — Contextual PheromoneWalker v2

From-scratch, no warm start. Same corrected SparseWalker v1.1 recurrence that won on Amazon, but behavioral learning uses **slow global pheromone + fast session pheromone + per-edge context prototypes + eligibility-trail credit**. Reward is tied directly to actual next-item ranking against deterministic negatives. No optimizer, no `backward()`, no autograd learning.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys, shutil, subprocess, runpy, json, torch
from pathlib import Path
REPO='/content/Sparsewalker'
BRANCH='agent/contextual-pheromone-walker-v2'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','-b',BRANCH,'https://github.com/hanialshater/Sparsewalker-.git',REPO], check=True)
for p in [f'{REPO}/src', f'{REPO}/experiments', f'{REPO}/benchmarks']:
    if p not in sys.path: sys.path.insert(0,p)
import sparsewalker
assert torch.cuda.is_available(), 'GPU runtime required'
print('GPU',torch.cuda.get_device_name(0),'torch',torch.__version__,'bf16',torch.cuda.is_bf16_supported())
print('BRANCH',BRANCH,'PACKAGE',sparsewalker.__file__)


## Run Beauty

The main comparison remains SASRec **0.03120** and gradient-trained SparseWalker v1.1 **0.04488** test NDCG@10. First pass uses 15 epochs; the best validation checkpoint is retained automatically.


In [ ]:
SCRIPT=f'{REPO}/experiments/run_amazon_contextual_pheromone_walker.py'
sys.argv=[SCRIPT,
    '--dataset','beauty',
    '--epochs','15',
    '--batch-size','512',
    '--eval-every','1',
]
runpy.run_path(SCRIPT, run_name='__main__')


## Diagnostics

Healthy behavior would show ranking reward and NDCG rising together. Watch whether context coverage grows without saturating immediately, whether rewiring stays sparse, and whether NDCG improves beyond PheromoneWalker v1's ~0.00235 validation peak.


In [ ]:
import pandas as pd
root=Path('/content/drive/MyDrive/sparsewalker_contextual_pheromone_v2/beauty/seed42')
hist=root/'history.json'
if hist.exists():
    df=pd.DataFrame(json.loads(hist.read_text()))
    cols=['epoch','val_NDCG@10','val_HR@10','mean_rank_reward','positive_reward_fraction','rewired_edges','context_edges_updated','slow_tau_mean','slow_tau_max','context_strength_mean','positions_per_s']
    display(df[[c for c in cols if c in df.columns]])
else:
    print('No history yet')


In [ ]:
res=root/'result.json'
if res.exists():
    r=json.loads(res.read_text())
    print(json.dumps(r, indent=2))
else:
    print('No result yet')


## What to send back

Please send `CPHERO_INIT`, the first 5–7 `CPHERO_EPOCH` rows, and `CPHERO_RESULT`. The key question is whether **actual ranking reward** now tracks validation quality instead of diverging from it.
